# Projet — Exploitation du dateset CL-Drive

## Estimation de la charge cognitive du conducteur

Ce TP vient après les TD1, TD2, TD3 et TD4.

Les TD ont déjà permis de travailler :

- la compréhension du papier CL-Drive ;
- le protocole expérimental ;
- la segmentation en fenêtres de 10 s ;
- le prétraitement EEG ;
- l'extraction des features EEG.

Le point de départ du TP est donc le dossier généré à la fin du TD4 :

```text
EEG_Features_10s/
```

Ce TP ne revient pas sur le calcul des features. Il exploite les features déjà extraites pour construire un pipeline d'apprentissage automatique.

## Objectif du TP

Construire un pipeline complet :

```text
EEG_Features_10s
→ Normalized_Features_10s/EEG
→ Normalized_Features_10s_With_Label/EEG
→ Dataset EEG supervisé
→ Classification de la charge cognitive
→ Évaluation
→ Interprétation
```

Dans un premier temps, on se limite à l'EEG uniquement.

La multimodalité, c'est-à-dire l'ajout de ECG, EDA et Gaze, sera proposée uniquement comme extension à la fin du sujet.

## 1. Structure attendue des dossiers

Avant de commencer, le dossier de travail doit contenir au minimum :

```text
Data/
├── EEG/ID_x
│   ├── ... fichiers level_1, level_2, ..., level_9
│   ├── ... fichiers baseline
│   └── ... fichiers filtered_*
│
├── EEG_Features_10s/
│   ├── ID1_EEG_features.csv
│   ├── ID2_EEG_features.csv
│   └── ...
│
├── Labels/
│   ├── ID1.csv
│   ├── ID2.csv
│   └── ...
```

Le TP va générer deux nouveaux dossiers :

```text
Data/
├── Normalized_Features_10s/
│   └── EEG/
│       ├── norm_ID1_EEG_features.csv
│       ├── norm_ID2_EEG_features.csv
│       └── ...
│
├── Normalized_Features_10s_With_Label/(avec colonnes Level et Label)
│   └── EEG/
│       ├── norm_ID1_EEG_features.csv
│       ├── norm_ID2_EEG_features.csv
│       └── ...
```

## Question 

Pourquoi ne faut-il pas entraîner directement les modèles sur les fichiers `EEG_Features_10s`  ?

### Réponse 

Les fichiers `EEG_Features_10s` contiennent des features brutes non normalisées et sans labels de charge cognitive. Les entraîner directement poserait deux problèmes : (1) les valeurs absolues varient fortement d'un sujet à l'autre (variabilité inter-individuelle), ce qui biaise les modèles vers les sujets à forte amplitude ; (2) les labels PAAS n'ont pas encore été associés à chaque fenêtre, il est donc impossible de formuler un problème de classification supervisée.

In [1]:
from pathlib import Path

# TODO : adapter ce chemin à votre organisation locale.
BASE_PATH = Path("../data")

EEG_FEATURE_DIR = BASE_PATH / "EEG_Features_10s"
LABEL_DIR = BASE_PATH / "Labels"
NORMALIZED_ROOT = BASE_PATH / "Normalized_Features_10s"
NORMALIZED_EEG_DIR = NORMALIZED_ROOT / "EEG"
LABELED_ROOT = BASE_PATH / "Normalized_Features_10s_With_Label"
LABELED_EEG_DIR = LABELED_ROOT / "EEG"

NORMALIZED_EEG_DIR.mkdir(parents=True, exist_ok=True)
LABELED_EEG_DIR.mkdir(parents=True, exist_ok=True)

METADATA_COLUMNS = ["subject", "scenario", "fenetre", "temps_debut", "temps_fin", "channel", "Level", "Label", "binary_class", "ternary_class"]

## 2. Normalisation des features EEG

À la fin du TD4, chaque fichier CSV contient des features EEG calculées sur des fenêtres de 10 secondes.

La normalisation doit suivre deux étapes :

### Étape 1 — Normalisation par la baseline du sujet

Pour chaque sujet, les fichiers de baseline servent à calculer une valeur moyenne de référence pour chaque feature :

$$
\mu_{baseline}^{(s,f)} = \frac{1}{N}\sum_{i=1}^{N} x_i^{(s,f)}
$$

où :

- $s$ désigne le sujet ;
- $f$ désigne la feature ;
- $x_i^{(s,f)}$ désigne la valeur de la feature pendant la baseline.

Chaque valeur de feature dans les fichiers de tâche est ensuite divisée par la moyenne de baseline correspondante :

$$
x_{norm}^{(s,f)} = \frac{ x^{(s,f)} }{ \mu_{baseline}^{(s,f)} }
$$

### Étape 2 — Standardisation z-score

On applique ensuite une standardisation :

$$
z = \frac{x - \mu}{\sigma}
$$

Cela permet d’obtenir des features centrées et réduites. Cette étape devra toutefois être réalisée plus loin dans le pipeline, après la séparation des données entre les ensembles d’entraînement et de test (voir section 8 ci-dessous).

## Question

Quel est l'intérêt de la normalisation par baseline dans des signaux physiologiques ?

### Réponse 

Les signaux physiologiques présentent une forte variabilité inter-individuelle : un même niveau de charge cognitive peut produire des amplitudes EEG très différentes d'un sujet à l'autre. En divisant par la moyenne baseline propre à chaque sujet, on ramène chaque valeur à une mesure relative (ratio par rapport au repos), ce qui rend les features comparables entre individus et aide le modèle à généraliser.

In [2]:
import pandas as pd

def get_feature_columns(df, metadata_columns=METADATA_COLUMNS):
    """
    Retourne les colonnes numériques correspondant aux features.
    Les colonnes de métadonnées ne doivent pas être normalisées.
    """
    # TODO : sélectionner les colonnes numériques et retirer les métadonnées.
    features = [c for c in df.columns if c not in metadata_columns]
    return df[features]


def compute_baseline_averages(feature_dir):
    baseline_avgs = {}
    feature_dir = Path(feature_dir)

    for file in feature_dir.rglob("*.csv"):
        df = pd.read_csv(file)

        if 'scenario' not in df.columns:
            continue

        baseline_df = df[df['scenario'].str.contains('baseline', na=False)]
        if baseline_df.empty:
            continue

        participant = baseline_df['subject'].iloc[0]
        feature_cols = get_feature_columns(baseline_df).columns
        baseline_avgs[participant] = baseline_df[feature_cols].mean()

    return baseline_avgs


def normalize_by_baseline(df, participant_id, baseline_avgs):
    """
    Divise chaque feature par sa moyenne de baseline pour le sujet considéré.
    """
    # TODO : compléter.
    df_norm = df.copy()
    feature_cols = get_feature_columns(df).columns
    baseline = baseline_avgs[participant_id]
    df_norm[feature_cols] = df[feature_cols] / baseline[feature_cols]
    return df_norm



def run_eeg_normalization():
    """
    Génère les fichiers du dossier :
    Normalized_Features_10s/EEG
    à partir du dossier :
    EEG_Features_10s
    """
    # TODO : calculer les profils baseline.
    # TODO : parcourir les fichiers non-baseline.
    # TODO : sauvegarder sous le nom norm_<nom_fichier>.csv dans NORMALIZED_EEG_DIR.
    feature_dir = Path(EEG_FEATURE_DIR)
    output_dir = Path(NORMALIZED_EEG_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    baseline_avgs = compute_baseline_averages(feature_dir)

    for file in feature_dir.rglob("*.csv"):
        if "baseline" in file.name:
            continue

        df = pd.read_csv(file)
        participant_id = df['subject'].iloc[0]

        if participant_id not in baseline_avgs:
            print(f"Pas de baseline pour {participant_id}, on passe.")
            continue

        df_norm = normalize_by_baseline(df, participant_id, baseline_avgs)

        out_file = output_dir / f"norm_{file.name}"
        df_norm.to_csv(out_file, index=False)
        print(f"Sauvegarde --> {out_file}")

    print("Normalisation terminée.")
    

In [68]:
run_eeg_normalization()

Sauvegarde --> ..\data\Normalized_Features_10s\EEG\norm_1030_EEG_features.csv
Sauvegarde --> ..\data\Normalized_Features_10s\EEG\norm_1105_EEG_features.csv
Sauvegarde --> ..\data\Normalized_Features_10s\EEG\norm_1106_EEG_features.csv
Sauvegarde --> ..\data\Normalized_Features_10s\EEG\norm_1241_EEG_features.csv
Sauvegarde --> ..\data\Normalized_Features_10s\EEG\norm_1271_EEG_features.csv
Sauvegarde --> ..\data\Normalized_Features_10s\EEG\norm_1314_EEG_features.csv
Sauvegarde --> ..\data\Normalized_Features_10s\EEG\norm_1323_EEG_features.csv
Sauvegarde --> ..\data\Normalized_Features_10s\EEG\norm_1337_EEG_features.csv
Sauvegarde --> ..\data\Normalized_Features_10s\EEG\norm_1372_EEG_features.csv
Sauvegarde --> ..\data\Normalized_Features_10s\EEG\norm_1417_EEG_features.csv
Sauvegarde --> ..\data\Normalized_Features_10s\EEG\norm_1434_EEG_features.csv
Sauvegarde --> ..\data\Normalized_Features_10s\EEG\norm_1544_EEG_features.csv
Sauvegarde --> ..\data\Normalized_Features_10s\EEG\norm_1547_EEG

## 3. Vérification du dossier `Normalized_Features_10s/EEG`

Après exécution de la normalisation, vérifiez que le dossier contient bien des fichiers `norm_*.csv`.

## Question

Pourquoi les fichiers de baseline ne sont-ils pas copiés dans le dossier normalisé final ?

### Réponse 

Les fichiers de baseline représentent l'état de repos du sujet et servent uniquement de référence pour calculer les facteurs de normalisation. Les inclure dans le dossier de sortie créerait une confusion dans les étapes suivantes (ajout des labels, construction du dataset) car ils ne correspondent à aucun niveau de scénario ni à aucun score PAAS.

In [3]:
# Vérification du dossier Normalized_Features_10s/EEG
norm_features_dir = Path(NORMALIZED_EEG_DIR)
for file in norm_features_dir.rglob("*"):
    assert file.name.startswith("norm_") and file.name.endswith(".csv")

print("Contenu du dossier Normalized_Features_10s/EEG ok !")

Contenu du dossier Normalized_Features_10s/EEG ok !


## 4. Ajout des colonnes `Level` et `Label`

Les fichiers normalisés ne contiennent pas encore la cible d'apprentissage.

Il faut maintenant associer chaque fenêtre de 10 secondes à son score PAAS.

Les labels sont stockés dans le dossier :

```text
Labels/
```

Chaque fichier de labels correspond à un sujet, par exemple :

```text
Labels/ID1.csv
Labels/ID2.csv
...
```

Dans ces fichiers, on suppose une structure du type :

| time | lvl_1 | lvl_2 | ... | lvl_9 |
|---:|---:|---:|---|---:|
| 10 | 2 | 3 | ... | 5 |
| 20 | 2 | 4 | ... | 6 |
| ... | ... | ... | ... | ... |

Pour une fenêtre d'indice `Window`, le temps associé est :

$$
time = (Window + 1) \times 10
$$

Le niveau du scénario est extrait du nom du fichier avec une expression régulière :

```text
level_1 → Level = 1
level_2 → Level = 2
...
level_9 → Level = 9
```

Le score PAAS est ensuite récupéré dans la colonne :

```text
lvl_<Level>
```

Exemple : si `Level = 4`, on lit la colonne `lvl_4`.


In [4]:
import re

def extract_level_from_filename(file_name):
    match = re.search(r"level_(\d+)", file_name)
    if match:
        return int(match.group(1))
    return None

def get_label_for_row(row, labels_df):
    window = row['fenetre']
    time_stamp = (window + 1) * 10
    level = extract_level_from_filename(row['scenario'])
    label_col = f'lvl_{level}'
    match = labels_df[labels_df['time'] == time_stamp]
    if match.empty or label_col not in match.columns:
        return None
    return match[label_col].iloc[0]

def attach_labels_eeg():
    input_dir = Path(NORMALIZED_EEG_DIR)
    output_dir = Path(LABELED_EEG_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    for file in input_dir.rglob("*.csv"):
        df = pd.read_csv(file)
        participant_id = df['subject'].iloc[0]

        labels_path = Path(LABEL_DIR) / f"{participant_id}.csv"
        if not labels_path.exists():
            print(f"Pas de labels pour {participant_id}, on passe.")
            continue

        labels_df = pd.read_csv(labels_path)

        df['Level'] = df['scenario'].apply(extract_level_from_filename)
        df['Label'] = df.apply(lambda row: get_label_for_row(row, labels_df), axis=1)

        df = df.dropna(subset=['Label'])

        out_file = output_dir / file.name
        df.to_csv(out_file, index=False)
        print(f"Sauvegarde -> {out_file}")

    print("Labelisation terminee.")

In [71]:
attach_labels_eeg()

C:\Users\racle\AppData\Local\Temp\ipykernel_32740\1891304925.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Level'] = df['scenario'].apply(extract_level_from_filename)
C:\Users\racle\AppData\Local\Temp\ipykernel_32740\1891304925.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Label'] = df.apply(lambda row: get_label_for_row(row, labels_df), axis=1)


Sauvegarde -> ..\data\Normalized_Features_10s_With_Label\EEG\norm_1030_EEG_features.csv


C:\Users\racle\AppData\Local\Temp\ipykernel_32740\1891304925.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Level'] = df['scenario'].apply(extract_level_from_filename)
C:\Users\racle\AppData\Local\Temp\ipykernel_32740\1891304925.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Label'] = df.apply(lambda row: get_label_for_row(row, labels_df), axis=1)


Sauvegarde -> ..\data\Normalized_Features_10s_With_Label\EEG\norm_1105_EEG_features.csv


C:\Users\racle\AppData\Local\Temp\ipykernel_32740\1891304925.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Level'] = df['scenario'].apply(extract_level_from_filename)
C:\Users\racle\AppData\Local\Temp\ipykernel_32740\1891304925.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Label'] = df.apply(lambda row: get_label_for_row(row, labels_df), axis=1)


...

C:\Users\racle\AppData\Local\Temp\ipykernel_32740\1891304925.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Level'] = df['scenario'].apply(extract_level_from_filename)
C:\Users\racle\AppData\Local\Temp\ipykernel_32740\1891304925.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Label'] = df.apply(lambda row: get_label_for_row(row, labels_df), axis=1)
C:\Users\racle\AppData\Local\Temp\ipykernel_32740\1891304925.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram

Sauvegarde -> ..\data\Normalized_Features_10s_With_Label\EEG\norm_1868_EEG_features.csv
Sauvegarde -> ..\data\Normalized_Features_10s_With_Label\EEG\norm_1892_EEG_features.csv


C:\Users\racle\AppData\Local\Temp\ipykernel_32740\1891304925.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Level'] = df['scenario'].apply(extract_level_from_filename)
C:\Users\racle\AppData\Local\Temp\ipykernel_32740\1891304925.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Label'] = df.apply(lambda row: get_label_for_row(row, labels_df), axis=1)
C:\Users\racle\AppData\Local\Temp\ipykernel_32740\1891304925.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram

Sauvegarde -> ..\data\Normalized_Features_10s_With_Label\EEG\norm_1953_EEG_features.csv
Labelisation terminee.


C:\Users\racle\AppData\Local\Temp\ipykernel_32740\1891304925.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Label'] = df.apply(lambda row: get_label_for_row(row, labels_df), axis=1)


## 5. Vérification du dossier `Normalized_Features_10s_With_Label/EEG`

Le dossier final doit contenir des fichiers CSV avec au moins :

- les métadonnées : `Participant`, `File`, `Window`, `Channel`, `Start_Time` et `End_Time` ;
- les features EEG normalisées ;
- la colonne `Level` ;
- la colonne `Label`.

## Question

Quelle est la différence entre `Level` et `Label` dans ce TP ? Pourquoi faut-il ajouter à la fois `Level` et `Label` ?

### Réponse

`Level` désigne le numéro du scénario de conduite (1 à 9), complexité objective imposée par les expérimentateurs. `Label` est le score PAAS déclaré par le conducteur (1 à 9), charge cognitive subjective ressentie. Ces deux informations sont distinctes : un scénario difficile ne garantit pas un score PAAS élevé pour tous les sujets. `Level` permet des analyses par scénario, `Label` constitue la cible de classification supervisée.

In [5]:
# Vérification du dossier Normalized_Features_10s_With_Label/EEG
REQUIRED_METADATA = ['subject', 'scenario', 'fenetre', 'temps_debut', 'temps_fin']
REQUIRED_EXTRA = ['Level', 'Label']

for file in Path(LABELED_EEG_DIR).rglob("*.csv"):
    df = pd.read_csv(file)

    # Vérification des colonnes obligatoires
    for col in REQUIRED_METADATA + REQUIRED_EXTRA:
        assert col in df.columns, f"{file.name} : colonne manquante '{col}'"

    # Vérification qu'il y a des features en plus des métadonnées
    feature_cols = get_feature_columns(df).columns
    assert len(feature_cols) > 0, f"{file.name} : aucune feature trouvee"

    # Vérification qu'il n'y a pas de Label manquant
    assert df['Label'].notna().all(), f"{file.name} : des lignes ont un Label NaN"

    # Vérification que Level est bien un entier entre 1 et 9
    assert df['Level'].between(1, 9).all(), f"{file.name} : des valeurs de Level sont hors bornes"

    print(f"OK : {file.name} ({len(df)} lignes, {len(feature_cols)} features)")

print("Tests ok !")

OK : norm_1030_EEG_features.csv (267 lignes, 160 features)
OK : norm_1105_EEG_features.csv (267 lignes, 160 features)
OK : norm_1106_EEG_features.csv (270 lignes, 160 features)
OK : norm_1241_EEG_features.csv (270 lignes, 160 features)
OK : norm_1271_EEG_features.csv (251 lignes, 160 features)
OK : norm_1314_EEG_features.csv (248 lignes, 160 features)
OK : norm_1323_EEG_features.csv (270 lignes, 160 features)
OK : norm_1337_EEG_features.csv (270 lignes, 160 features)
OK : norm_1372_EEG_features.csv (270 lignes, 160 features)
OK : norm_1417_EEG_features.csv (270 lignes, 160 features)
OK : norm_1434_EEG_features.csv (260 lignes, 160 features)
OK : norm_1544_EEG_features.csv (259 lignes, 160 features)
OK : norm_1547_EEG_features.csv (245 lignes, 160 features)
OK : norm_1595_EEG_features.csv (270 lignes, 160 features)
OK : norm_1629_EEG_features.csv (210 lignes, 160 features)
OK : norm_1716_EEG_features.csv (263 lignes, 160 features)
OK : norm_1717_EEG_features.csv (261 lignes, 160 feature

## 6. Construction du dataset EEG supervisé

Une fois les fichiers normalisés et labellisés générés, on peut les concaténer pour construire un tableau unique.

Chaque ligne représente une fenêtre EEG de 10 secondes pour un canal.

On construit ensuite deux problèmes possibles :

### Classification binaire

| Score PAAS | Classe |
|---:|---|
| 1 à 4 | faible |
| 5 à 9 | élevée |

### Classification ternaire, extension

| Score PAAS | Classe |
|---:|---|
| 1 à 3 | faible |
| 4 à 6 | moyenne |
| 7 à 9 | élevée |

Dans ce TP, l'objectif principal est la classification binaire.

In [6]:
def load_labeled_eeg_dataset():
    dfs = []
    for file in Path(LABELED_EEG_DIR).rglob("*.csv"):
        df = pd.read_csv(file)
        df = df[~df['scenario'].str.contains('baseline', na=False)]
        dfs.append(df)
    
    dataset = pd.concat(dfs, ignore_index=True)
    
    dataset['binary_class'] = dataset['Label'].apply(lambda x: 'faible' if x <= 4 else 'elevee')
    dataset['ternary_class'] = dataset['Label'].apply(lambda x: 'faible' if x <= 3 else ('moyenne' if x <= 6 else 'elevee'))
    
    return dataset


df = load_labeled_eeg_dataset()
print(df.shape)
df.head()

(3138, 169)


C:\Users\racle\AppData\Local\Temp\ipykernel_28804\1964843330.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset['binary_class'] = dataset['Label'].apply(lambda x: 'faible' if x <= 4 else 'elevee')
C:\Users\racle\AppData\Local\Temp\ipykernel_28804\1964843330.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dataset['ternary_class'] = dataset['Label'].apply(lambda x: 'faible' if x <= 3 else ('moyenne' if x <= 6 else 'elevee'))


,subject,scenario,fenetre,temps_debut,temps_fin,AF7_psd_delta_sum,AF7_psd_delta_mean,AF7_psd_delta_max,AF7_psd_delta_min,AF7_psd_delta_median,...,TP10_raw_mean,TP10_raw_min,TP10_raw_max,TP10_raw_median,TP10_raw_var,TP10_raw_std,Level,Label,binary_class,ternary_class
0,1030,eeg_data_level_1,0,120.007812,130.003906,1.246879,1.246879,1.167543,1.385791,1.762491,...,0.938980,0.931875,0.660968,1.026249,0.519192,0.753492,1,2.0,faible,faible
1,1030,eeg_data_level_1,1,130.007812,140.003906,3.693992,3.693992,3.451984,4.797900,3.224579,...,0.900974,1.196111,0.822782,0.864210,1.013674,1.052844,1,2.0,faible,faible
2,1030,eeg_data_level_1,2,140.007812,150.003906,37.275049,37.275049,25.557130,85.660788,45.894233,...,1.117285,1.740439,3.516020,1.161282,3.795163,2.037183,1,3.0,faible,faible
3,1030,eeg_data_level_1,3,150.007812,160.003906,0.680771,0.680771,0.979648,0.725144,0.614591,...,1.048300,1.152072,0.715820,0.891217,1.129597,1.111416,1,2.0,faible,faible
4,1030,eeg_data_level_1,4,160.007812,170.003906,16.470063,16.470063,19.450122,15.660012,13.487539,...,0.888771,1.113317,2.920875,0.972236,1.843963,1.420008,1,2.0,faible,faible


In [7]:
df["binary_class"] = df["Label"].apply(lambda x: 0 if x <= 4 else 1)
df["ternary_class"] = df["Label"].apply(lambda x: 0 if x <= 3 else (1 if x <= 6 else 2))

## 7. Préparation de la matrice d'apprentissage

On doit séparer :

- les métadonnées ;
- les features numériques EEG ;
- la cible d'apprentissage.

## Question

Pourquoi ne faut-il pas inclure `Participant`, `File`, `Window`, `Level` ou `Label` dans les features du modèle ?

### Réponse 

Ces colonnes ne sont pas des mesures physiologiques. Inclure `sujet` entraînerait le modèle à reconnaître les individus plutôt que la charge ; inclure `Level` lui fournirait la complexité imposée du scénario, non disponible en conditions réelles ; inclure `Label` constituerait une fuite directe de la cible.

In [8]:
# Préparation des données d'entraînement
X = get_feature_columns(df)
y = df["binary_class"]

print("Nombre d'exemples :", len(X))
print("Nombre de features :", X.shape[1])
print("Exemples de features :", list(X.columns[:5]))

Nombre d'exemples : 3138
Nombre de features : 160
Exemples de features : ['AF7_psd_delta_sum', 'AF7_psd_delta_mean', 'AF7_psd_delta_max', 'AF7_psd_delta_min', 'AF7_psd_delta_median']


## 8. Classification EEG — premiers modèles

On teste plusieurs modèles classiques :

- LDA ;
- SVM ;
- Random Forest ;
- KNN ;
- Naive Bayes ;
- Decision Tree ;
- AdaBoost ;
- MLP.

La normalisation `StandardScaler` est placée dans le `sklearn.pipeline.Pipeline` pour éviter une fuite de données entre apprentissage et test. Il faut ajuster le `StandardScaler` uniquement sur les données d’entraînement :

`scaler.fit_transform(X_train)`

Puis appliquer la transformation aux données de test avec :

`scaler.transform(X_test)`

In [18]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "LDA":           LinearDiscriminantAnalysis(),
    "SVM":           SVC(),
    "Random Forest": RandomForestClassifier(),
    "KNN":           KNeighborsClassifier(),
    "Naive Bayes":   GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(),
    "AdaBoost":      AdaBoostClassifier(),
    "MLP":           MLPClassifier(max_iter=500),
}

results = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    results[name] = acc
    print(f"{name:15} -> accuracy : {acc:.4f} / F1 score : {f1}")

print("\nMeilleur modele :", max(results, key=results.get))

LDA             -> accuracy : 0.6879 / F1 score : 0.7270194986072424
SVM             -> accuracy : 0.7373 / F1 score : 0.7767253044654939
Random Forest   -> accuracy : 0.7962 / F1 score : 0.825136612021858
KNN             -> accuracy : 0.7771 / F1 score : 0.7971014492753623
Naive Bayes     -> accuracy : 0.5669 / F1 score : 0.5903614457831325
Decision Tree   -> accuracy : 0.6656 / F1 score : 0.6938775510204082
AdaBoost        -> accuracy : 0.7277 / F1 score : 0.7729083665338645
MLP             -> accuracy : 0.7564 / F1 score : 0.7823613086770982

Meilleur modele : Random Forest


## 9. Évaluation par validation croisée et par sujet

Deux évaluations sont demandées :

### 10-fold cross-validation

Les segments sont répartis en 10 folds stratifiés. Cette évaluation est utile pour comparer les modèles, mais elle peut mélanger les sujets entre apprentissage et test.

### Leave-One-Subject-Out, LOSO

Un sujet est laissé de côté pour le test, tandis que le modèle est entraîné sur les autres sujets. Cette stratégie d’évaluation est plus réaliste, car elle permet de tester la capacité de généralisation du modèle sur un conducteur jamais vu auparavant. L’opération est ensuite répétée sur l’ensemble des sujets disponibles afin d’obtenir une évaluation plus robuste.

## Question

Pourquoi le LOSO est-il souvent plus difficile que le 10-fold classique ?

### Réponse 

Le LOSO est plus difficile car le modèle doit généraliser à un sujet complètement inconnu. En 10-fold classique, des fenêtres du même sujet se retrouvent dans le train et dans le test — le modèle peut s'appuyer sur des patterns propres à l'individu. En LOSO, ces patterns individuels ne sont d'aucune aide puisque le sujet de test n'a jamais été vu.

In [15]:
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut, cross_val_score

groups = df['subject'].values

# 10-fold cross-validation stratifiée
print("=== 10-fold Cross-Validation ===")
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

results_kfold = {}
for name, model in models.items():
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", model)
    ])
    scores = cross_val_score(pipeline, X, y, cv=skf, scoring='accuracy')
    f1_scores = cross_val_score(pipeline, X, y, cv=skf, scoring='f1')
    results_kfold[name] = scores.mean()
    print(f"{name:15} -> accuracy : {scores.mean():.4f} (+/- {scores.std():.4f}) / F1 score : {f1_scores.mean()}")

# LOSO
print("\n=== Leave-One-Subject-Out ===")
logo = LeaveOneGroupOut()

results_loso = {}
for name, model in models.items():
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", model)
    ])
    scores = cross_val_score(pipeline, X, y, cv=logo, groups=groups, scoring='accuracy')
    f1_scores = cross_val_score(pipeline, X, y, cv=logo, groups=groups, scoring='f1')
    results_loso[name] = scores.mean()
    print(f"{name:15} -> accuracy : {scores.mean():.4f} (+/- {scores.std():.4f}) / F1 score : {f1_scores.mean()}")

print("\nMeilleur modele (10-fold) :", max(results_kfold, key=results_kfold.get))
print("Meilleur modele (LOSO)   :", max(results_loso,  key=results_loso.get))

=== 10-fold Cross-Validation ===
LDA             -> accuracy : 0.7221 (+/- 0.0391) / F1 score : 0.7697230386404877
SVM             -> accuracy : 0.7651 (+/- 0.0275) / F1 score : 0.8104947352136372
Random Forest   -> accuracy : 0.7992 (+/- 0.0236) / F1 score : 0.8362660058284792
KNN             -> accuracy : 0.7773 (+/- 0.0239) / F1 score : 0.8064137367287367
Naive Bayes     -> accuracy : 0.4761 (+/- 0.0170) / F1 score : 0.28758628583041335
Decision Tree   -> accuracy : 0.6861 (+/- 0.0352) / F1 score : 0.7297188258794594
AdaBoost        -> accuracy : 0.7374 (+/- 0.0322) / F1 score : 0.783749555908849
MLP             -> accuracy : 0.7779 (+/- 0.0174) / F1 score : 0.8157137851847477

=== Leave-One-Subject-Out ===
LDA             -> accuracy : 0.6303 (+/- 0.1383) / F1 score : 0.6523692638636108
SVM             -> accuracy : 0.6220 (+/- 0.1571) / F1 score : 0.6657216000251308
Random Forest   -> accuracy : 0.5838 (+/- 0.1704) / F1 score : 0.6256969130656167
KNN             -> accuracy : 0.57

In [11]:
print(y.value_counts())

binary_class
1    1812
0    1326
Name: count, dtype: int64


## 10. Interprétation et discussion

Répondez aux questions suivantes dans le notebook :

1. Quel modèle obtient le meilleur F1-score en 10-fold ? **Random Forest**
2. Quel modèle obtient le meilleur F1-score en LOSO ? **SVM**
3. Les performances chutent-elles en LOSO ? Pourquoi ? **Oui, typiquement de 5 à 15 points de F1. La normalisation par baseline atténue la variabilité inter-individuelle mais ne l'élimine pas : le modèle n'a jamais vu les patterns EEG du sujet testé.**
4. Les classes sont-elles équilibrées ? **1812 pour 1, et 1326 pour 0, donc c'est plutôt équilibré.**
5. Les résultats obtenus avec EEG seul vous semblent-ils suffisants pour une application réelle ? **Des F1 > 0.70 sont encourageants, mais une application embarquée exigerait une robustesse bien supérieure et une faible latence.**
6. Quelles limites voyez-vous à l'utilisation des labels subjectifs PAAS ? **Auto-rapportés et donc subjectifs, interprétation variable entre individus, fatigue de labellisation, granularité à 9 niveaux difficile à distinguer finement.**
7. Quelles améliorations proposeriez-vous ?
   - **Fusionner EEG + ECG + EDA + Gaze (multimodalité)** ;
   - **Appliquer une ICA pour supprimer les artefacts oculaires et musculaires** ;
   - **Explorer CNN 1D ou LSTM sur les signaux bruts** ;
   - **Transfer learning ou fine-tuning pour l'adaptation au sujet.**

## 11. Mini-système d'adaptation

À partir de la prédiction du modèle, on peut simuler une décision d'adaptation.

Exemple :

| Prédiction | Décision |
|---|---|
| charge faible | interface normale |
| charge élevée | simplification de l'interface |
| charge élevée persistante | alerte conducteur |

## Question 

Pourquoi faut-il être prudent avant de déclencher une alerte sur une seule prédiction ?

### Réponse 

Une prédiction isolée peut être un faux positif dû à un artefact transitoire (clignement, mouvement de tête). Déclencher une alerte immédiate risque de perturber le conducteur inutilement, augmentant paradoxalement sa charge cognitive. Il faut un mécanisme de lissage temporel (fenêtre glissante, seuil sur N prédictions consécutives) avant toute décision critique.

In [9]:
def decision_system(predictions, n_consecutive=3):
    """
    Transforme les prédictions en decision d'adaptation.
    
    predictions   : liste ou array de predictions binaires (0=faible, 1=elevee)
    n_consecutive : nombre de predictions elevees consecutives pour declencher une alerte
    """
    if len(predictions) == 0:
        return "interface normale"

    last_pred = predictions[-1]

    # Verifier si les n dernieres predictions sont toutes elevees
    if len(predictions) >= n_consecutive:
        last_n = predictions[-n_consecutive:]
        if all(p == 1 for p in last_n):
            return "alerte conducteur"

    if last_pred == 1:
        return "simplification de l'interface"
    else:
        return "interface normale"


# Exemple d'utilisation
predictions_simulees = [0, 0, 1, 1, 1]
print(decision_system(predictions_simulees))  # -> alerte conducteur

predictions_simulees = [0, 0, 1, 0, 1]
print(decision_system(predictions_simulees))  # -> simplification de l'interface

predictions_simulees = [0, 0, 0, 0, 0]
print(decision_system(predictions_simulees))  # -> interface normale

alerte conducteur
simplification de l'interface
interface normale


## 12. Extension optionnelle — vers la multimodalité

Le cœur du TP est volontairement limité à l'EEG.

Une extension possible consiste à reproduire les mêmes étapes pour les autres modalités :

```text
ECG_Features_10s → Normalized_Features_10s/ECG → Normalized_Features_10s_With_Label/ECG
EDA_Features_10s → Normalized_Features_10s/EDA → Normalized_Features_10s_With_Label/EDA
Gaze_Features_10s → Normalized_Features_10s/Gaze → Normalized_Features_10s_With_Label/Gaze
```

Puis à fusionner les features :

```text
EEG + ECG
EEG + EDA
EEG + Gaze
EEG + ECG + EDA + Gaze
```

La fusion la plus simple est une concaténation des colonnes de features pour des fenêtres correspondant au même sujet, au même niveau et au même indice de fenêtre.

## Question

Pourquoi la multimodalité peut-elle améliorer la détection de la charge cognitive ?

### Réponse 

Chaque modalité capture un aspect différent : l'EEG reflète l'activité neuronale directe, l'ECG mesure la variabilité cardiaque, l'EDA traduit l'activation sympathique, le Gaze révèle les stratégies visuelles. Ces signaux sont partiellement complémentaires : quand l'EEG est bruité par un artefact, l'EDA peut compenser. La fusion permet au modèle d'exploiter une représentation plus robuste de l'état cognitif, comme le confirment les résultats du papier CL-Drive où la combinaison de toutes les modalités donne les meilleurs scores.